# Library Inclusion and Utilities

<!-- structured-notebook -->
## Notebook Summary
Purpose: prototype a retrieval-augmented LangGraph workflow that summarizes article text, predicts whether it is relevant to healthy aging or longevity, retrieves supporting examples when confidence is low, and then reclassifies the item.

Main steps:
- Load the Chroma vector store and define the state object carried through the graph.
- Implement graph nodes for summarization, initial classification, retrieval, and reclassification.
- Wire the graph together and run a small test slice to inspect the resulting behavior.


In [ ]:
# structured-notebook-bootstrap
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


_repo_root = find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.append(str(_repo_root))

from src.project_paths import (
    ARXIV_RAW_DIR,
    CHROMA_DIR,
    EXTERNAL_NEWS_DIR,
    GUARDIAN_DATA_DIR,
    LLM_CLASSIFICATION_DIR,
    NEWS_HTML_DIR,
    NEWS_OUTPUT_DIR,
    PREPRINT_RAW_DIR,
    PROQUEST_PROCESSED_DIR,
    PROQUEST_UNPROCESSED_DIR,
    PUBMED_PROCESSED_DIR,
    PUBMED_RAW_DIR,
    PUBLICATIONS_TABLE_DIR,
    REDDIT_DATA_DIR,
    ROOT,
    RQ1_FIGURES_DIR,
    RQ4_PLOTS_DIR,
    TOPIC_MATCHING_DIR,
    YOUTUBE_DATA_DIR,
)


In [1]:
import pandas as pd
import numpy as np
import itables.options as opt
from itables import show
import matplotlib.pyplot as plt
import requests, json
import re, time
from typing import TypedDict, Optional

## Helpers

In [31]:
def show_row(df:pd.DataFrame,i):
    row = df.loc[i]
    print("TOPIC:", row["BERTopic_topic"])
    print("\nTITLE:\n", row["Title"])
    print("\nFULL TEXT:\n", row["Full text"])
    print("\nLLM VERDICT:\n", row["llm_verdict"])
    print("\nRELATED_LANG:\n", row["Related_Lang"])
    print("\nsummary_Lang:\n", row["summary_Lang"])
    print("\nevidence_Lang:\n", row["evidence_Lang"])
    print("\nsupporting_docs_Lang:\n", row["supporting_docs_Lang"])

# Sentence Tranformer and Vector Database

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection("news_articles")

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# Defining Object Class

In [4]:
class State(TypedDict):
    text: str
    topic_name: str
    summary: Optional[str]
    verdict: Optional[dict]
    retrieved: Optional[str]
    retrieved_ids: Optional[list]

# Preparing Test Input

Topic number: topic name
- 11: Dementia_alzheimers
- 12: food
- 14: retirement
- 27: physical activity
- 33: intermittent fasting
- 44: blue zones
- 48: sleeping
- 57: menopause, hrt
- 63: green tea/ black tea/ coffee

In [5]:
def first_n_sentences(text, n=5):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return " ".join(sentences[:n])

In [6]:
test_df = pd.read_csv(LLM_CLASSIFICATION_DIR / "bertopic_test3_df.csv")
test_df['llm_input'] = test_df['Title'] + test_df['Full text'].apply(first_n_sentences)

In [7]:
food_df = test_df[test_df['BERTopic_topic']==12].copy()

<!-- structured-notebook -->
## Node Implementation Order
The next blocks define the LangGraph nodes in the same order they are expected to run: summarize first, classify next, retrieve evidence if needed, then reclassify with that evidence in hand.


# Nodes

## Node 1

In [8]:
from langchain_community.llms import Ollama

llm = Ollama(model="qwen2.5:7b-instruct", temperature=0)

def summarize_node(state: State) -> State:
    prompt = f"Summarize in 2 sentences:\n\n{state['text']}"
    summary = llm.invoke(prompt)
    state["summary"] = summary
    return state

/tmp/ipykernel_54389/2066480810.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="qwen2.5:7b-instruct", temperature=0)


## Node 2

In [ ]:
import json, re

def extract_json(out: str):
    try:
        return json.loads(out)
    except Exception:
        m = re.search(r"\{.*\}", out, re.DOTALL)
        if not m:
            print("NO JSON FOUND. RAW OUTPUT:\n", out)
            raise
        js = m.group(0)
        try:
            return json.loads(js)
        except Exception as e:
            print("FAILED JSON:\n", js)
            print("RAW OUTPUT:\n", out)
            raise e


def classify_node(state: State) -> State:
    topic = state["topic_name"]
    prompt = f"""
Return ONLY JSON.

Decide if the summary is related to Healthy Aging/Longevity in terms of subtopic: {topic}

JSON format:
{{"Related_RAG":"true|false|N/A","summary_RAG":"...","confidence_RAG":1}}

Text:
{state["summary"]}
""".strip()

    out = llm.invoke(prompt)
    state["verdict"] = extract_json(out)
    return state


## Node 3

In [10]:
def retrieve_node(state: State) -> State:
    topic = state["topic_name"]
    summary = state["summary"]

    query_text = f"Subtopic: {topic}\nText: {summary}"

    q_emb = embedder.encode([query_text], normalize_embeddings=True).tolist()

    res = collection.query(
        query_embeddings=q_emb,
        n_results=3
    )

    retrieved_docs = res["documents"][0]  
    retrieved_ids = res["ids"][0]        

    state["retrieved"] = "\n\n---\n\n".join(retrieved_docs)
    state["retrieved_ids"] = retrieved_ids

    return state

## Node 4

In [22]:
def reclassify_node(state: State) -> State:
    topic = state["topic_name"]

    prompt = f"""
Return ONLY valid JSON. No markdown. No extra text.

Task:
Decide whether the TEXT is related to Healthy Aging/Longevity in terms of the subtopic: {topic}

Rules:
- Use the retrieved docs as evidence.
- If you decide "true", your evidence must quote or clearly reference a retrieved doc.
- supporting_doc_id must be one of the provided Retrieved IDs, or "N/A".
- evidence must be one sentence and must not contain any double quote characters (").
- If you need to refer to a term, use single quotes.

Return JSON in EXACTLY this format:
{{
  "Related_Lang": "true|false|N/A",
  "summary_Lang": "one short sentence",
  "confidence_Lang": "Between 0 and 10",
  "evidence_Lang": "short justification based on retrieved docs",
  "supporting_doc_id": "id from Retrieved IDs or N/A"
}}

Retrieved IDs:
{state.get("retrieved_ids", [])}

Retrieved docs:
{state["retrieved"]}

Text:
{state["summary"]}
""".strip()

    out = llm.invoke(prompt)
    state["verdict"] = extract_json(out)
    return state

<!-- structured-notebook -->
## Graph Assembly And Trial Run
This final section connects the nodes, defines the routing rule, and executes a small example so the pipeline can be inspected end to end.


# Conditional Routing and Graph Building

In [23]:
def needs_retrieval(state: State):
    conf = state["verdict"].get("confidence", 0)
    return "retrieve" if conf < 6 else "end"

In [24]:
from langgraph.graph import StateGraph, END

graph = StateGraph(State)

graph.add_node("summarize", summarize_node)
graph.add_node("classify", classify_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("reclassify", reclassify_node)

graph.set_entry_point("summarize")

graph.add_edge("summarize", "classify")

graph.add_conditional_edges(
    "classify",
    needs_retrieval,
    {"retrieve": "retrieve", "end": END}
)

graph.add_edge("retrieve", "reclassify")
graph.add_edge("reclassify", END)

app = graph.compile()

In [14]:
result = app.invoke({
    "text": food_df.loc[389, "llm_input"],
    "topic_name": "food",
    "summary": None,
    "verdict": None,
    "retrieved": None
})

'A recent survey by the International Food Information Council shows that 90% of Americans now recognize "functional" foods—those that offer health benefits beyond basic nutrition—and understand how specific foods like yogurt can target particular health aspects such as bone health and gut health. This increased awareness is attributed to scientific studies highlighting the preventive health benefits of certain foods, marking a significant shift from previous years when only about three-quarters of Americans could make similar associations.'

In [ ]:
food_df = test_df[test_df['BERTopic_topic']==12].copy()

In [27]:
food_df = food_df[0:5].copy()

In [28]:
i = 0
for row in food_df.iterrows():
    result = app.invoke({
        "text": row[1]['llm_input'],
        "topic_name": "food",
        "summary": None,
        "verdict": None,
        "retrieved": None
    })
    food_df["Related_Lang"] = result["verdict"]["Related_Lang"]
    food_df["summary_Lang"] = result["verdict"]["summary_Lang"]
    food_df["confidence_Lang"] = result["verdict"]["confidence_Lang"]
    food_df["evidence_Lang"] = result["verdict"]["evidence_Lang"]
    food_df["supporting_doc_id_Lang"] = result["verdict"]["supporting_doc_id"]
    food_df["supporting_docs_Lang"] = result["retrieved"]
    i+=1
    print(i)

1
2
3
4
5


In [29]:
food_df

,Title,Abstract,Full text,Author,Subject,Publication title,Publication year,Publication date,Section,Publisher,...,Related,summary,confidence,llm_input,Related_Lang,summary_Lang,confidence_Lang,evidence_Lang,supporting_doc_id_Lang,supporting_docs_Lang
389,"'Food can act as medicine' -- and now, Americ...",[...] Americans are much more aware of the he...,"Say you eat yogurt for your health, and mos...","connolly, gregory",Nutrition; Functional foods & nutraceuticals;...,"USA TODAY; McLean, Va.",2011,"Aug 4, 2011",LIFE,"USA Today, a division of Gannett Satellite In...",...,True,The text discusses the awareness and consumpti...,8.0,"'Food can act as medicine' -- and now, Americ...",true,The text is related to food as it mentions Rob...,8,He disclosed that he might have contracted the...,doc_48153,"India, Nov. 17 -- Robert F Kennedy Junior, a ..."
390,"Love the 30s!: So what if you're 30, don't ...",[...] if you're getting into a fitness regime...,"If you're pushing 30, feeling 'old' should be...","talukdar, taniya",Nutrition; Stress; Sports training; Pregnancy...,DNA : Daily News & Analysis; Mumbai,2011,"Jan 25, 2011",LIFESTYLE & LEISURE,"Diligent Media Corporation, Ltd., DNA - Resea...",...,True,The text discusses dietary recommendations for...,9.0,"Love the 30s!: So what if you're 30, don't ...",true,The text is related to food as it mentions Rob...,8,He disclosed that he might have contracted the...,doc_48153,"India, Nov. 17 -- Robert F Kennedy Junior, a ..."
391,A chocolate bar a day is not sweet for your l...,None available.,A daily chocolate bar could be causing you to...,NaN,Diet; Food; Aging; Calories,"The Daily Telegraph; Surry Hills, N.S.W.",2024,"Dec 10, 2024",News,Nationwide News Pty Ltd,...,True,The text discusses the impact of ultra-process...,8.0,A chocolate bar a day is not sweet for your l...,true,The text is related to food as it mentions Rob...,8,He disclosed that he might have contracted the...,doc_48153,"India, Nov. 17 -- Robert F Kennedy Junior, a ..."
392,"'Brain worms from India?': RFK Jr, Trump's he...",None available.,"In 2010, Robert F. Kennedy Jr. experienced se...",NaN,Political campaigns; Personal health; Online ...,The Economic Times; New Delhi,2024,"Nov 18, 2024",NaN,"Bennett, Coleman & Company Limited",...,False,The text does not directly discuss food relate...,3.0,"'Brain worms from India?': RFK Jr, Trump's he...",true,The text is related to food as it mentions Rob...,8,He disclosed that he might have contracted the...,doc_48153,"India, Nov. 17 -- Robert F Kennedy Junior, a ..."
393,"RFK Jr, Donald Trump's public health appointe...",None available.,"India, Nov. 17 -- Robert F Kennedy Junior, a ...",NaN,Public health; Cognitive ability; Social netw...,The Hindustan Times; New Delhi,2024,"Nov 17, 2024",NaN,HT Digital Streams Limited,...,True,The text discusses Robert F Kennedy Jr.'s diet...,8.0,"RFK Jr, Donald Trump's public health appointe...",true,The text is related to food as it mentions Rob...,8,He disclosed that he might have contracted the...,doc_48153,"India, Nov. 17 -- Robert F Kennedy Junior, a ..."


In [32]:
show_row(food_df, 389)

TOPIC: 12.0

TITLE:
  'Food can act as medicine' -- and now, Americans know it:   90% can identify 'functional' fare

FULL TEXT:
    Say you eat yogurt for your health, and most Americans will know what you mean: You are targeting that food's bone-building calcium and gut-friendly probiotics. In fact, Americans are much more aware of the health benefits of specific "functional" foods than they were a decade ago, a survey reports today. When the International Food Information Council began its survey in 1998, "only about three-fourths of Americans could name a food and its related health benefits," says the group's Elizabeth Rahavi. "Now, almost nine out of 10 can. A lot has to do with scientific studies coming out, talking about the benefits of a food and its relationship to good health." The IFIC defines functional foods as "foods or food components that may provide benefits beyond basic nutrition." "People are 1,000% more conscious of the fact that food can act as medicine and help p